In [0]:
# Objective: Provide a df containing 'order_id' and 'contact_address' that should adhere to the following information and format: "city name, postal code". If the city name is not available, the placeholder "Unknown" should be used. If the postal code is not known, the placeholder "UNK00" should be used.

import re
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import sys
sys.path.insert(0, "/Workspace/IFCO_Challenge/src")
from transformations import get_address

spark = SparkSession.builder.getOrCreate()
 
ORDERS_PATH     = "/Volumes/workspace/ifco_test/ifco_resources/orders.csv"
INVOICING_PATH  = "/Volumes/workspace/ifco_test/ifco_resources/invoicing_data.json"

df_orders_raw = (
    spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("escape", '"')
    .csv(ORDERS_PATH)
)

In [0]:

# Custom function is used to apply the transformations
get_address_udf = F.udf(get_address, StringType())

df_2 = (
    df_orders_raw
    .withColumn("contact_address", get_address_udf(F.col("contact_data")))
    .select("order_id", "contact_address")
)

df_2.show(truncate=False)

# Quick count check
total    = df_2.count()
nulos    = df_2.filter(F.col("contact_address").isNull()).count()
unknown  = df_2.filter(F.col("contact_address").startswith("Unknown")).count()
unk00    = df_2.filter(F.col("contact_address").endswith("UNK00")).count()

print(f"Total count:{total}")
print(f"Null adress count:{nulos}")
print(f"City placeholder count: {unknown}")
print(f"CP placeholder count: {unk00}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/udf.py:103: UserWarning: Cannot infer the eval type from type hints. 
  warnings.warn("Cannot infer the eval type from type hints. ", UserWarning)


+------------------------------------+-----------------------------+
|order_id                            |contact_address              |
+------------------------------------+-----------------------------+
|f47ac10b-58cc-4372-a567-0e02b2c3d479|Chicago, 12345               |
|f47ac10b-58cc-4372-a567-0e02b2c3d480|Calcutta, UNK00              |
|f47ac10b-58cc-4372-a567-0e02b2c3d481|Frankfurt am Oder, 3934      |
|f47ac10b-58cc-4372-a567-0e02b2c3d482|Unknown, UNK00               |
|f47ac10b-58cc-4372-a567-0e02b2c3d483|Unknown, UNK00               |
|f47ac10b-58cc-4372-a567-0e02b2c3d484|New York, 1203               |
|f47ac10b-58cc-4372-a567-0e02b2c3d485|Unknown, UNK00               |
|f47ac10b-58cc-4372-a567-0e02b2c3d486|Esplugues de Llobregat, UNK00|
|f47ac10b-58cc-4372-a567-0e02b2c3d487|Tel Aviv, UNK00              |
|f47ac10b-58cc-4372-a567-0e02b2c3d488|Chicago, 12345               |
|f47ac10b-58cc-4372-a567-0e02b2c3d489|Barcelona, 8023              |
|f47ac10b-58cc-4372-a567-0e02b2c3d

In [0]:
# Guardamos en Unity Catalog
df_2.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("workspace.ifco_test.address")